# 2025-26 Prompt Experiments — Sarcasm, /s, and Slang Lines

**Objective:** Decide the frozen v2 classification prompt (#62 item 3) by running
candidate prompt lines through the production harness
(`classify_cases(prompt_builder=...)`) against the 100-case eval suite. Decision
metric, per the ticket: sarcasm gains vs. regression on genuine praise of
struggling players — adopt only if net-positive. Deliverable: this notebook +
per-variant result artifacts under `data/2025-26/reference/prompt_experiments/`,
an adopt/reject verdict per line, and the frozen prompt applied to
`pipeline/batch.py` (same PR, separate commit). Floors and `known_miss` flags in
`tests/eval/cases.yaml` are re-pinned only after the freeze.

**This notebook answers:**

- §1 — Control: does a fresh single run of the production prompt reproduce the
  pinned 07-22 baseline?
- §2 — `/s`-hint line: does the classifier read the marker when told to?
  (sarcasm-m19 keeps its tag intact and still misses at baseline.)
- §3 — The ticket's sarcasm line, as written (directional: ironic praise → neg).
- §4 — A direction-neutral sarcasm line. The baseline over-negs 4 of 7
  sarcasm-neu cases; §3's "classify as negative" push cuts against those.
- §5 — Slang line: decision record — the item-2 audit already settled it
  (no change; the adopt list is nominated by the classifier's own correct
  outputs, and the suite has no case where those terms carry sentiment).
- §6 — Freeze candidate (production + `/s`-hint), 3-run confirmed.
- §7 — Verdict: per-line adopt/reject with suite evidence; the frozen v2 prompt.


## Load Data

Guardrails:

- **Every variant run costs ~100 sync Messages API calls** (Haiku @ temp 0,
  ~$0.02/run). Results persist to `EXP_DIR` keyed by a sha of the built prompt:
  re-executing the notebook re-spends nothing until a prompt actually changes,
  and a changed prompt invalidates only its own artifact.
- **Variants live here, not in `pipeline/batch.py`.** Production `build_prompt`
  is imported untouched as the §1 control; only the §7-frozen winner lands in
  `pipeline/batch.py`. The pytest suite keeps measuring production throughout.
- **Scoring is the harness's own:** production model params and parse path
  (`classify_cases`), `accuracy_by_category`, attribution via the alias map.
  Single-run deltas are readable because the 07-22 baseline had zero flaky
  outcomes across 3 runs; the freeze candidate still gets 3 runs in §6.


In [1]:
import hashlib
import json
import os
import sys
from collections.abc import Callable
from pathlib import Path

# Bootstrap: run from anywhere (see 01–05 — walk up to the repo root, chdir).
_root = Path.cwd()
while not (_root / "pyproject.toml").exists() and _root != _root.parent:
    _root = _root.parent
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import polars as pl

from pipeline.batch import MODEL, TEMPERATURE, build_prompt
from pipeline.evaluation import (
    accuracy_by_category,
    classify_cases,
    load_cases,
    load_category_floors,
)
from utils.paths import get_data_dir
from utils.season_config import get_active_season

SEASON = "2025-26"  # pinned: this notebook is season-scoped (notebooks/2025-26/)
assert get_active_season() == SEASON, "active season flipped — review before re-running"

EXP_DIR = get_data_dir(season=SEASON) / "reference" / "prompt_experiments"
EXP_DIR.mkdir(parents=True, exist_ok=True)

CASES = load_cases()
FLOORS = load_category_floors()
CASE_BY_ID = {c.id: c for c in CASES}

# The pinned 07-22 3-run baseline, derived from the known_miss flags (a case
# is expected-correct iff it isn't flagged). Fresh runs compare against this.
PINNED = {
    cat: (
        sum(1 for c in CASES if c.category == cat and not c.known_miss),
        sum(1 for c in CASES if c.category == cat),
    )
    for cat in FLOORS
}

assert os.environ.get("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY not set"
print(f"{len(CASES)} cases, model {MODEL} @ temp {TEMPERATURE}")
print(
    pl.DataFrame(
        [
            {"category": cat, "pinned": f"{c}/{t}", "floor": FLOORS[cat]}
            for cat, (c, t) in sorted(PINNED.items())
        ]
    )
)


100 cases, model claude-haiku-4-5-20251001 @ temp 0.0
shape: (9, 3)
┌───────────────────┬────────┬───────┐
│ category          ┆ pinned ┆ floor │
│ ---               ┆ ---    ┆ ---   │
│ str               ┆ str    ┆ f64   │
╞═══════════════════╪════════╪═══════╡
│ clear_negative    ┆ 6/6    ┆ 0.83  │
│ clear_positive    ┆ 5/5    ┆ 0.8   │
│ genuine_praise    ┆ 29/29  ┆ 0.96  │
│ multi_player      ┆ 5/9    ┆ 0.44  │
│ negative_nickname ┆ 3/3    ┆ 1.0   │
│ neutral           ┆ 3/3    ┆ 1.0   │
│ sarcasm           ┆ 5/20   ┆ 0.2   │
│ short             ┆ 18/21  ┆ 0.8   │
│ slang_inversion   ┆ 4/4    ┆ 1.0   │
└───────────────────┴────────┴───────┘


In [2]:
# Variant runner + scoring helpers used by every section below.


def prompt_sha(prompt_builder: Callable[[str], str]) -> str:
    """A variant's identity: sha of the prompt built for a sentinel body."""
    return hashlib.sha256(prompt_builder("__CANARY__").encode()).hexdigest()[:16]


def run_variant(
    name: str, prompt_builder: Callable[[str], str], run: int = 1
) -> dict[str, dict]:
    """Classify all cases under a variant, cached on (name, run, prompt sha).

    Bump `run` for an independent repeat of the same prompt (stability
    checks); a changed prompt under the same name re-runs automatically.
    """
    sha = prompt_sha(prompt_builder)
    artifact = EXP_DIR / f"{name}_run{run}.json"
    if artifact.exists():
        payload = json.loads(artifact.read_text())
        if payload["prompt_sha"] == sha:
            return payload["results"]
        print(f"{artifact.name}: prompt changed (was {payload['prompt_sha']}) — re-running")
    results = classify_cases(CASES, prompt_builder=prompt_builder)
    artifact.write_text(
        json.dumps({"prompt_sha": sha, "model": MODEL, "results": results}, indent=1)
    )
    errors = sum(1 for r in results.values() if r["s"] == "error")
    print(f"{artifact.name}: written ({errors} parse errors)")
    return results


def category_table(results: dict[str, dict]) -> pl.DataFrame:
    """Per-category sentiment accuracy vs the pinned baseline and the floor."""
    tallies = accuracy_by_category(CASES, results)
    return pl.DataFrame(
        [
            {
                "category": cat,
                "acc": f"{c}/{t}",
                "pinned": f"{pc}/{pt}",
                "delta": c - pc,
                "floor_ok": c / t >= FLOORS[cat],
            }
            for cat, (c, t) in sorted(tallies.items())
            for pc, pt in [PINNED[cat]]
        ]
    ).sort("delta")


FLIP_SCHEMA = {
    "id": str, "category": str, "expected": str,
    "ref": str, "got": str, "flip": str, "text": str,
}


def flips(results: dict[str, dict], against: dict[str, dict]) -> pl.DataFrame:
    """Cases whose sentiment correctness changed vs a reference run.

    The per-case review artifact: `fixed` rows are the variant's wins,
    `BROKEN` rows are its cost. Same-wrong-answer changes don't appear.
    """
    rows = []
    for case in CASES:
        got, ref = results[case.id]["s"], against[case.id]["s"]
        if (got == case.expected) == (ref == case.expected):
            continue
        rows.append(
            {
                "id": case.id, "category": case.category, "expected": case.expected,
                "ref": ref, "got": got,
                "flip": "fixed" if got == case.expected else "BROKEN",
                "text": case.text[:60],
            }
        )
    return pl.DataFrame(rows, schema=FLIP_SCHEMA).sort(["flip", "category", "id"])


## 1. Control — fresh run of the production prompt

`PINNED` above is the 07-22 3-run picture reconstructed from the `known_miss`
flags. A fresh run that matches it makes every later single-run delta readable
as prompt effect rather than drift. Per-case drift (a fresh miss on an
unflagged case, or a surprise pass on a flagged one) prints below — an empty
table is a clean reproduction, and this run becomes the flip-table reference
for every variant.


In [3]:
# §1 — control run: production build_prompt, untouched.
baseline = run_variant("baseline", build_prompt)
print(category_table(baseline))

drift = pl.DataFrame(
    [
        {
            "id": c.id, "category": c.category, "expected": c.expected,
            "got": baseline[c.id]["s"],
            "drift": "surprise pass" if c.known_miss else "fresh miss",
            "text": c.text[:60],
        }
        for c in CASES
        if (baseline[c.id]["s"] == c.expected) == c.known_miss
    ],
    schema={
        "id": str, "category": str, "expected": str,
        "got": str, "drift": str, "text": str,
    },
)
print(f"\nper-case drift vs pinned flags: {drift.height} case(s)")
print(drift)


shape: (9, 5)
┌───────────────────┬───────┬────────┬───────┬──────────┐
│ category          ┆ acc   ┆ pinned ┆ delta ┆ floor_ok │
│ ---               ┆ ---   ┆ ---    ┆ ---   ┆ ---      │
│ str               ┆ str   ┆ str    ┆ i64   ┆ bool     │
╞═══════════════════╪═══════╪════════╪═══════╪══════════╡
│ clear_negative    ┆ 6/6   ┆ 6/6    ┆ 0     ┆ true     │
│ clear_positive    ┆ 5/5   ┆ 5/5    ┆ 0     ┆ true     │
│ genuine_praise    ┆ 29/29 ┆ 29/29  ┆ 0     ┆ true     │
│ multi_player      ┆ 5/9   ┆ 5/9    ┆ 0     ┆ true     │
│ negative_nickname ┆ 3/3   ┆ 3/3    ┆ 0     ┆ true     │
│ neutral           ┆ 3/3   ┆ 3/3    ┆ 0     ┆ true     │
│ sarcasm           ┆ 5/20  ┆ 5/20   ┆ 0     ┆ true     │
│ short             ┆ 18/21 ┆ 18/21  ┆ 0     ┆ true     │
│ slang_inversion   ┆ 4/4   ┆ 4/4    ┆ 0     ┆ true     │
└───────────────────┴───────┴────────┴───────┴──────────┘

per-case drift vs pinned flags: 0 case(s)
shape: (0, 6)
┌─────┬──────────┬──────────┬─────┬───────┬──────┐
│ id  ┆ 

## 2. `/s`-hint line

The narrowest intervention: one line telling the classifier the marker exists.
Baseline evidence says the model doesn't read `/s` at all — sarcasm-m19 keeps
its tag intact and still scores `neu @ 0.5`. Only that one case is directly in
scope (the other 19 sarcasm cases are tag-stripped by design, mirroring
production where `/s` is rare: 2,427 of 2.15M comments), so the win condition
is m19 fixed with an empty `BROKEN` column — not a sarcasm jump. The flip
table also answers a side question: does merely *mentioning* sarcasm in the
prompt move untagged cases?

The factory below composes variant prompts in the production shape (swappable
slang line, optional extra lines between slang and the comment) and is
asserted byte-identical to production `build_prompt` at defaults — every
later section builds its candidate through it.


In [4]:
# §2 — prompt factory (asserted faithful to production) + the /s-hint line.
PROMPT_HEAD = "Classify sentiment toward NBA players."
PROD_SLANG_LINE = (
    "Slang: nasty/sick/filthy=positive, washed/brick/fraud/cooked=negative, "
    "GOAT=positive."
)
PROMPT_TAIL = (
    'Respond ONLY with JSON: {"s":"pos|neg|neu","c":0.0-1.0,"p":"Player Name"|null}'
)


def make_builder(
    slang_line: str = PROD_SLANG_LINE, extra: tuple[str, ...] = ()
) -> Callable[[str], str]:
    """Variant prompts in the production shape; defaults reproduce it exactly."""

    def builder(comment_body: str) -> str:
        head = "\n".join([PROMPT_HEAD, slang_line, *extra])
        return f"{head}\n\nComment: {comment_body}\n\n{PROMPT_TAIL}"

    return builder


assert make_builder()("__x__") == build_prompt("__x__"), "factory drifted from production"

S_HINT_LINE = 'A trailing "/s" tags the comment as sarcasm.'

s_hint = run_variant("s_hint", make_builder(extra=(S_HINT_LINE,)))
print(category_table(s_hint))
print(flips(s_hint, baseline))

_m19 = s_hint["sarcasm-m19"]
print(f"\nsarcasm-m19 (tag intact): expected neg, got {_m19['s']} @ {_m19['c']}")


shape: (9, 5)
┌───────────────────┬───────┬────────┬───────┬──────────┐
│ category          ┆ acc   ┆ pinned ┆ delta ┆ floor_ok │
│ ---               ┆ ---   ┆ ---    ┆ ---   ┆ ---      │
│ str               ┆ str   ┆ str    ┆ i64   ┆ bool     │
╞═══════════════════╪═══════╪════════╪═══════╪══════════╡
│ multi_player      ┆ 4/9   ┆ 5/9    ┆ -1    ┆ true     │
│ clear_negative    ┆ 6/6   ┆ 6/6    ┆ 0     ┆ true     │
│ clear_positive    ┆ 5/5   ┆ 5/5    ┆ 0     ┆ true     │
│ genuine_praise    ┆ 29/29 ┆ 29/29  ┆ 0     ┆ true     │
│ negative_nickname ┆ 3/3   ┆ 3/3    ┆ 0     ┆ true     │
│ neutral           ┆ 3/3   ┆ 3/3    ┆ 0     ┆ true     │
│ short             ┆ 18/21 ┆ 18/21  ┆ 0     ┆ true     │
│ slang_inversion   ┆ 4/4   ┆ 4/4    ┆ 0     ┆ true     │
│ sarcasm           ┆ 8/20  ┆ 5/20   ┆ 3     ┆ true     │
└───────────────────┴───────┴────────┴───────┴──────────┘
shape: (4, 7)
┌─────────────┬──────────────┬──────────┬─────┬─────┬────────┬─────────────────────────────────┐
│ id 

## 3. The ticket's sarcasm line, as written

#62's candidate: *"Sarcasm is common. If praise seems ironic given the
player's known weaknesses, classify as negative."* Directional by
construction — it targets the 8 expected-neg sarcasm misses and pushes
against the 7 sarcasm-neu cases the model already over-negs (4 of 7 missed as
`neg` at baseline) and the 4 sarcasm-pos cases. First real test of the
genuine_praise trap: 29/29 with a 0.96 floor means two regressions fail the
suite. Token cost if adopted: ~20 tokens ≈ $22 per 2.15M-comment run.


In [5]:
# §3 — the ticket's sarcasm line, verbatim from #62.
TICKET_SARCASM_LINE = (
    "Sarcasm is common. If praise seems ironic given the player's known "
    "weaknesses, classify as negative."
)

sarcasm_ticket = run_variant("sarcasm_ticket", make_builder(extra=(TICKET_SARCASM_LINE,)))
print(category_table(sarcasm_ticket))
print(flips(sarcasm_ticket, baseline))


shape: (9, 5)
┌───────────────────┬───────┬────────┬───────┬──────────┐
│ category          ┆ acc   ┆ pinned ┆ delta ┆ floor_ok │
│ ---               ┆ ---   ┆ ---    ┆ ---   ┆ ---      │
│ str               ┆ str   ┆ str    ┆ i64   ┆ bool     │
╞═══════════════════╪═══════╪════════╪═══════╪══════════╡
│ genuine_praise    ┆ 27/29 ┆ 29/29  ┆ -2    ┆ false    │
│ short             ┆ 17/21 ┆ 18/21  ┆ -1    ┆ true     │
│ clear_negative    ┆ 6/6   ┆ 6/6    ┆ 0     ┆ true     │
│ clear_positive    ┆ 5/5   ┆ 5/5    ┆ 0     ┆ true     │
│ multi_player      ┆ 5/9   ┆ 5/9    ┆ 0     ┆ true     │
│ negative_nickname ┆ 3/3   ┆ 3/3    ┆ 0     ┆ true     │
│ neutral           ┆ 3/3   ┆ 3/3    ┆ 0     ┆ true     │
│ slang_inversion   ┆ 4/4   ┆ 4/4    ┆ 0     ┆ true     │
│ sarcasm           ┆ 9/20  ┆ 5/20   ┆ 4     ┆ true     │
└───────────────────┴───────┴────────┴───────┴──────────┘
shape: (7, 7)
┌─────────────┬────────────────┬──────────┬─────┬───────┬────────┬─────────────────────────────────┐
│

## 4. Direction-neutral sarcasm line

§3's autopsy gives this line its two constraints: no player-specific "known
weaknesses" framing (that's what flipped genuine praise of Gobert's meme
weakness), and terse/declarative wording (§3's reasoning-style instruction
invited reasoning in the output — two JSON-contract breaks). Candidate:
*"Sarcasm and mock praise are common; judge the commenter's actual attitude,
not the literal words."* Direction-neutral on purpose: the sarcasm category
is only 9/20 expected-neg — 7 cases are neu and 4 are pos, and a line that
only pushes toward neg caps out on those. ~19 tokens ≈ $21/run if adopted.


In [6]:
# §4 — direction-neutral sarcasm line.
NEUTRAL_SARCASM_LINE = (
    "Sarcasm and mock praise are common; judge the commenter's actual "
    "attitude, not the literal words."
)

sarcasm_neutral = run_variant("sarcasm_neutral", make_builder(extra=(NEUTRAL_SARCASM_LINE,)))
print(category_table(sarcasm_neutral))
print(flips(sarcasm_neutral, baseline))

_errors = [cid for cid, r in sarcasm_neutral.items() if r["s"] == "error"]
print(f"\nJSON-contract breaks: {_errors or 'none'}")


shape: (9, 5)
┌───────────────────┬───────┬────────┬───────┬──────────┐
│ category          ┆ acc   ┆ pinned ┆ delta ┆ floor_ok │
│ ---               ┆ ---   ┆ ---    ┆ ---   ┆ ---      │
│ str               ┆ str   ┆ str    ┆ i64   ┆ bool     │
╞═══════════════════╪═══════╪════════╪═══════╪══════════╡
│ genuine_praise    ┆ 26/29 ┆ 29/29  ┆ -3    ┆ false    │
│ clear_negative    ┆ 6/6   ┆ 6/6    ┆ 0     ┆ true     │
│ clear_positive    ┆ 5/5   ┆ 5/5    ┆ 0     ┆ true     │
│ multi_player      ┆ 5/9   ┆ 5/9    ┆ 0     ┆ true     │
│ negative_nickname ┆ 3/3   ┆ 3/3    ┆ 0     ┆ true     │
│ neutral           ┆ 3/3   ┆ 3/3    ┆ 0     ┆ true     │
│ short             ┆ 18/21 ┆ 18/21  ┆ 0     ┆ true     │
│ slang_inversion   ┆ 4/4   ┆ 4/4    ┆ 0     ┆ true     │
│ sarcasm           ┆ 8/20  ┆ 5/20   ┆ 3     ┆ true     │
└───────────────────┴───────┴────────┴───────┴──────────┘
shape: (8, 7)
┌─────────────┬────────────────┬──────────┬─────┬───────┬────────┬─────────────────────────────────┐
│

## 5. Slang line — no change (decision, not experiment)

The item-2 audit already answered this: no genuinely new 2025-26 slang exists
(frequency-ratio surfaces roster churn, not language), and the top-10 adopt
list (cinema, lfg, "dawg in him", hooper, bum, poverty, phantom, flopper,
underrated, backhanded superlatives) was nominated by log-odds over *v1
classified data* — i.e., by the classifier's own consistent handling of those
terms across 1.93M comments. The evidence that nominated each term is the
evidence the model doesn't need it in the prompt: the existing slang line was
written against observed v1-era failures, and no adopt-list term has an
observed failure behind it. The suite agrees — none of the ten terms appears
in any case as a sentiment carrier, so a slang-line variant could show harm
here but structurally no benefit.

**Decision: slang line unchanged.** ~$15–20/run of input tokens not spent on
terms the model demonstrably handles. The backhanded-superlative pattern
("GOAT of disappearing in the playoffs") is a sarcasm-shaped question, not a
coverage one, and §3–§4 already recorded where prompt-side sarcasm handling
lands: marker-based yes, judgment-based no. No API calls spent here.


## 6. Freeze candidate — production + `/s`-hint, 3-run confirmation

The surviving delta is one line. §3–§4 rejected judgment-based sarcasm
instructions (both fail the genuine_praise floor and break the JSON
contract); §5 left the slang line unchanged. Three runs settle what §2's
single run couldn't: whether the untagged sarcasm fixes (m07, and the
always-flaky sarcasm-02) are stable prompt effects, and whether the
multi-m05 break is a real cost or noise. The stable-flip table at the bottom
is the `cases.yaml` re-pin driver — every stable change of correctness vs
the pinned flags becomes a `known_miss` add/remove in the freeze commit.


In [7]:
# §6 — freeze candidate: 3 runs of production + /s-hint (run 1 is §2's, cached).
FREEZE_BUILDER = make_builder(extra=(S_HINT_LINE,))

runs = [run_variant("s_hint", FREEZE_BUILDER, run=k) for k in (1, 2, 3)]
tallies = [accuracy_by_category(CASES, r) for r in runs]

print(
    pl.DataFrame(
        [
            {
                "category": cat,
                **{f"run{k}": f"{t[cat][0]}/{t[cat][1]}" for k, t in enumerate(tallies, 1)},
                "pinned": f"{PINNED[cat][0]}/{PINNED[cat][1]}",
                "floor_ok_all": all(t[cat][0] / t[cat][1] >= FLOORS[cat] for t in tallies),
            }
            for cat in sorted(FLOORS)
        ]
    )
)

_unstable = [
    {
        "id": c.id, "category": c.category, "expected": c.expected,
        "got": "/".join(r[c.id]["s"] for r in runs), "text": c.text[:60],
    }
    for c in CASES
    if len({r[c.id]["s"] == c.expected for r in runs}) > 1
]
print(f"\nunstable cases across 3 runs: {len(_unstable)}")
print(pl.DataFrame(_unstable, schema={
    "id": str, "category": str, "expected": str, "got": str, "text": str,
}))

# Stable correctness changes vs the pinned flags -> the cases.yaml re-pin.
_repin = [
    {
        "id": c.id, "category": c.category, "expected": c.expected,
        "got": runs[0][c.id]["s"],
        "action": "remove known_miss" if c.known_miss else "add known_miss",
        "text": c.text[:60],
    }
    for c in CASES
    if len({r[c.id]["s"] == c.expected for r in runs}) == 1
    and (runs[0][c.id]["s"] == c.expected) == c.known_miss
]
print(f"\nstable flips vs pinned flags (cases.yaml re-pin): {len(_repin)}")
print(pl.DataFrame(_repin, schema={
    "id": str, "category": str, "expected": str,
    "got": str, "action": str, "text": str,
}))


shape: (9, 6)
┌───────────────────┬───────┬───────┬───────┬────────┬──────────────┐
│ category          ┆ run1  ┆ run2  ┆ run3  ┆ pinned ┆ floor_ok_all │
│ ---               ┆ ---   ┆ ---   ┆ ---   ┆ ---    ┆ ---          │
│ str               ┆ str   ┆ str   ┆ str   ┆ str    ┆ bool         │
╞═══════════════════╪═══════╪═══════╪═══════╪════════╪══════════════╡
│ clear_negative    ┆ 6/6   ┆ 6/6   ┆ 6/6   ┆ 6/6    ┆ true         │
│ clear_positive    ┆ 5/5   ┆ 5/5   ┆ 5/5   ┆ 5/5    ┆ true         │
│ genuine_praise    ┆ 29/29 ┆ 29/29 ┆ 29/29 ┆ 29/29  ┆ true         │
│ multi_player      ┆ 4/9   ┆ 4/9   ┆ 5/9   ┆ 5/9    ┆ true         │
│ negative_nickname ┆ 3/3   ┆ 3/3   ┆ 3/3   ┆ 3/3    ┆ true         │
│ neutral           ┆ 3/3   ┆ 3/3   ┆ 3/3   ┆ 3/3    ┆ true         │
│ sarcasm           ┆ 8/20  ┆ 8/20  ┆ 8/20  ┆ 5/20   ┆ true         │
│ short             ┆ 18/21 ┆ 18/21 ┆ 18/21 ┆ 18/21  ┆ true         │
│ slang_inversion   ┆ 4/4   ┆ 4/4   ┆ 4/4   ┆ 4/4    ┆ true         │
└─────

## 7. Verdict & freeze

| Line | Decision | Evidence (artifacts in `EXP_DIR`) |
|---|---|---|
| `/s`-hint | **adopt** | sarcasm 5→8/20 stable ×3 runs; genuine_praise 29/29 ×4 runs; 0 JSON breaks in 300 calls; ~11 tokens ≈ $12/run |
| Ticket's sarcasm line | **reject** | genuine_praise 27/29 — floor fail (praise-m29: "known weaknesses" framing flips genuine praise of a meme weakness); 2 JSON-contract breaks |
| Direction-neutral sarcasm line | **reject** | genuine_praise 26/29 — worse ("actual attitude" framing licenses second-guessing sincere praise); 1 JSON break |
| Slang line | **no change** | §5: item-2 audit — no new 25-26 slang; adopt list nominated by the classifier's own consistent outputs over 1.93M v1 comments |

The frozen v2 prompt is production + one line. Teaching notation beat teaching
judgment: both judgment-based lines paid for sarcasm gains out of the
genuine_praise floor and invited prose that broke the JSON contract; the
marker line held every floor with spillover to untagged sarcasm (m07, 02).
Known limitation, accepted for v2: the line says *trailing* `/s`, and only
the trailing form is evidenced (m19); mid-text tags exist but are rarer.
The sarcasm ceiling that remains (8/20) is context-bound, not prompt-bound —
parent-comment context is a v3 question.

**Freeze sequence** (this notebook executes first, then):

1. `tests/eval/cases.yaml` — prune praise-m04 (ruled out 07-15 as cheeky
   trade humour, never removed; 28 praise cases, 27/28 ≈ 0.964 keeps the
   0.96 floor); remove `known_miss` from sarcasm-02/m07/m19 (notes credit
   the `/s` line); add `known_miss` to multi-m05 (unstable under the v2
   prompt: neu/neu/neg across §6 runs); floors: sarcasm 0.2 → 0.35
   (8/20 stable, minus slack), multi_player 0.44 → 0.33 (4/9 with m05
   flagged, minus slack).
2. `pipeline/batch.py` — `build_prompt` gains the line.
3. `uv run pytest -m eval --maxfail=0 -rxX` ×3 — green under the new
   prompt, flags, and floors.
4. After the freeze, this notebook is a closed record: §2's faithfulness
   assert fails against the new production prompt *by design*. The executed
   outputs and `EXP_DIR` artifacts are the evidence; do not re-execute.


In [8]:
# §7 — the frozen prompt, printed from the winning builder.
frozen_prompt = FREEZE_BUILDER("{comment_body}")
print(frozen_prompt)
print(f"\nfreeze sha: {prompt_sha(FREEZE_BUILDER)}  "
      f"(matches s_hint_run[1-3].json artifacts)")
print(f"delta vs production: +{len(S_HINT_LINE)} chars, 1 line: {S_HINT_LINE!r}")


Classify sentiment toward NBA players.
Slang: nasty/sick/filthy=positive, washed/brick/fraud/cooked=negative, GOAT=positive.
A trailing "/s" tags the comment as sarcasm.

Comment: {comment_body}

Respond ONLY with JSON: {"s":"pos|neg|neu","c":0.0-1.0,"p":"Player Name"|null}

freeze sha: d9515f31c0361ea6  (matches s_hint_run[1-3].json artifacts)
delta vs production: +44 chars, 1 line: 'A trailing "/s" tags the comment as sarcasm.'


### Post-freeze addendum: attribution stability

`pytest -m eval` against the frozen prompt caught what §6 didn't measure:
sentiment stability was confirmed ×3, but the `p` field was never checked, and
the `/s`-hint moved it. From the cached §6 artifacts (no new runs needed):

| Case | v1 prompt | v2 prompt ×3 runs | Re-pin |
|---|---|---|---|
| short-m04 | `null` (flagged) | `Kevin Durant` ×3 — stable-correct | flag removed |
| short-m11 | `null` (flagged) | `None/None/Bronny` — still flaky | unchanged |
| multi-m05 | correct | `None/None/Russ` — flaky, parallel to its sentiment wobble | `known_miss_player` added |
| multi-m18 | `Nikola Jokic` (correct) | `Dort` ×3 — **stable regression** | `known_miss_player` added, item-4 evidence |

Net attribution effect of the line: +1 stable fix, −1 stable regression, one
case wobbling in each direction — sentiment-neutral, and further evidence
that multi-mention `p` is unreliable (the item-4 write-up's running theme).
Methodology note for the next experiment notebook: `flips()` should cover
both axes, not just `s`.
